# Feature Selection + Retrain (Smaller Feature Sets)

This notebook continues from **feature_importance notebook**.

## Goal
**Top-40** may still be too many features, so here we run a focused sweep on smaller feature sets:

- **K = [40, 35, 30, 25, 20]** (top-K features from `selected_features_ranked.txt`)

We **select on Validation only** (to avoid test leakage) and then:

1) pick the **smallest K** within a tolerance of the best score  
2) evaluate once on **Test** (report only)  
3) retrain final model on **Train+Val** and save artifacts

## Decision policy (multiclass 0/1/2)
We use the **two-threshold rule**:

1. If `P(class2) >= tau2` → predict **2**  
2. Else if `P(class1) >= tau1` → predict **1**  
3. Else → predict **0**

## Outputs saved to `../models/`
- `feature_selection_results.csv` (and `feature_selection_results_partial.csv` checkpoint)
- `catboost_selected_top{K}.cbm`
- `catboost_selected_top{K}_thresholds.json`
- `catboost_selected_top{K}_features.txt`


In [1]:
# ============================================================
# 0) Imports + paths
# ============================================================
from __future__ import annotations

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.metrics import recall_score, precision_score, confusion_matrix
from catboost import CatBoostClassifier

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SUPERVISED_PATH = DATA_DIR / "supervised_hood_3h_multiclass.csv"
RANK_PATH = MODEL_DIR / "selected_features_ranked.txt"   # created in notebook 09

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Dataset:", SUPERVISED_PATH)
print("Rank file:", RANK_PATH, "exists:", RANK_PATH.exists())


Dataset: ../data/processed/supervised_hood_3h_multiclass.csv
Rank file: ../models/selected_features_ranked.txt exists: True


In [2]:
# ============================================================
# 1) Load dataset + split (time-based)
# ============================================================
df = pd.read_csv(SUPERVISED_PATH, low_memory=False)
df["time_3h"] = pd.to_datetime(df["time_3h"], errors="coerce")
df = df.dropna(subset=["time_3h"]).copy()

if "HOOD_158_CODE" in df.columns:
    df["HOOD_158_CODE"] = df["HOOD_158_CODE"].astype(str).str.zfill(3)

y = df["y_class"].astype(int)

drop_cols = [c for c in ["y_class", "y_count_next", "time_3h"] if c in df.columns]
X = df.drop(columns=drop_cols)

# IMPORTANT: keep same split you used earlier
train_mask = df["time_3h"] <= pd.Timestamp("2024-12-31 23:59:59")
val_mask   = (df["time_3h"] >= pd.Timestamp("2025-01-01")) & (df["time_3h"] <= pd.Timestamp("2025-06-30 23:59:59"))
test_mask  = df["time_3h"] >= pd.Timestamp("2025-07-01")

X_train, y_train = X.loc[train_mask].copy(), y.loc[train_mask].copy()
X_val,   y_val   = X.loc[val_mask].copy(),   y.loc[val_mask].copy()
X_test,  y_test  = X.loc[test_mask].copy(),  y.loc[test_mask].copy()

print("Shapes:")
print("  Train:", X_train.shape, y_train.shape)
print("  Val  :", X_val.shape, y_val.shape)
print("  Test :", X_test.shape, y_test.shape)

print("\nVal distribution:")
print(y_val.value_counts(normalize=True).sort_index().round(4))


Shapes:
  Train: (922720, 47) (922720,)
  Val  : (228784, 47) (228784,)
  Test : (232418, 47) (232418,)

Val distribution:
y_class
0    0.8951
1    0.0907
2    0.0141
Name: proportion, dtype: float64


In [3]:
# ============================================================
# 2) Column types
# ============================================================
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
if "HOOD_158_CODE" in X_train.columns and "HOOD_158_CODE" not in cat_cols:
    cat_cols.append("HOOD_158_CODE")

num_cols = [c for c in X_train.columns if c not in cat_cols]

print("Num cols:", len(num_cols))
print("Cat cols:", len(cat_cols))
print("Example cat cols:", cat_cols[:10])


Num cols: 46
Cat cols: 1
Example cat cols: ['HOOD_158_CODE']


## Helpers: metrics + two-threshold tuning

Selection happens on **Validation** using:
- `macro_recall` (primary)
- plus a small bonus for `precision_high` (class 2 precision)

Constraints (you can change):
- `min_recall_1 >= 0.25` (keep class 1 useful)
- `max_pred2_rate <= 0.15` (limit too many class-2 alerts)


In [4]:
def eval_multiclass(y_true, y_pred, name="model"):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1,2])
    rec = recall_score(y_true, y_pred, labels=[0,1,2], average=None, zero_division=0)
    macro = recall_score(y_true, y_pred, average="macro", zero_division=0)

    y_true_high = (y_true == 2).astype(int)
    y_pred_high = (y_pred == 2).astype(int)
    recall_high = recall_score(y_true_high, y_pred_high, zero_division=0)
    precision_high = precision_score(y_true_high, y_pred_high, zero_division=0)

    out = {
        "macro_recall": float(macro),
        "recall_high": float(recall_high),
        "precision_high": float(precision_high),
        "recall_0": float(rec[0]),
        "recall_1": float(rec[1]),
        "recall_2": float(rec[2]),
        "pred2_rate": float((np.asarray(y_pred) == 2).mean()),
    }
    print(f"\n=== {name} ===")
    print("Confusion matrix [0,1,2]:\n", cm)
    print("Metrics:", out)
    return out

def apply_two_thresholds(proba, tau1, tau2):
    p1 = proba[:, 1]
    p2 = proba[:, 2]
    pred = np.zeros(len(p1), dtype=int)
    pred[p1 >= tau1] = 1
    pred[p2 >= tau2] = 2
    return pred

def tune_two_thresholds(
    proba_val, y_val,
    tau1_grid=np.linspace(0.20, 0.70, 11),
    tau2_grid=np.linspace(0.05, 0.40, 15),
    min_recall_1=0.25,
    max_pred2_rate=0.15
):
    rows = []
    for tau1 in tau1_grid:
        for tau2 in tau2_grid:
            pred = apply_two_thresholds(proba_val, tau1, tau2)
            rec = recall_score(y_val, pred, labels=[0,1,2], average=None, zero_division=0)
            macro = recall_score(y_val, pred, average="macro", zero_division=0)
            rec2 = recall_score((y_val==2).astype(int), (pred==2).astype(int), zero_division=0)
            prec2 = precision_score((y_val==2).astype(int), (pred==2).astype(int), zero_division=0)
            pred2_rate = float((pred==2).mean())
            rows.append((tau1, tau2, macro, rec[1], rec2, prec2, pred2_rate))

    df_grid = pd.DataFrame(rows, columns=["tau1","tau2","macro_recall","recall_1","recall_2","precision_2","pred2_rate"])
    feasible = df_grid[(df_grid["recall_1"] >= min_recall_1) & (df_grid["pred2_rate"] <= max_pred2_rate)]

    best = (feasible if len(feasible) else df_grid).sort_values(
        ["macro_recall","recall_2","precision_2"], ascending=False
    ).iloc[0]

    return float(best["tau1"]), float(best["tau2"]), df_grid


## Load ranked features (from previous Notebook)


In [5]:
if not RANK_PATH.exists():
    raise FileNotFoundError(f"Rank file not found: {RANK_PATH}. Run 09_feature_importance first.")

with open(RANK_PATH, "r", encoding="utf-8") as f:
    ranked_features = [line.strip() for line in f if line.strip()]

ranked_features = [f for f in ranked_features if f in X_train.columns]

print("Ranked features available:", len(ranked_features))
print("Top 20:", ranked_features[:20])


Ranked features available: 47
Top 20: ['HOOD_158_CODE', 'hour_cos', 'hour_sin', 'block_hour', 'is_weekend', 'dow_num', 'coll_roll_mean_8', 'snow', 'rain', 'wind_speed_lag_1', 'coll_lag_2', 'snow_on_ground_lag_1', 'pd_collisions_lag_1', 'coll_roll_sum_4', 'coll_roll_sum_8', 'ftr_collisions_lag_1', 'visibility', 'snow_on_ground', 'coll_lag_1', 'injury_collisions_lag_1']


## Training function (CatBoost)

Stable settings for this project:
- `loss_function="MultiClassOneVsAll"`  
- `eval_metric="TotalF1:average=Macro"`  
- `auto_class_weights="SqrtBalanced"`  


In [6]:
def train_catboost_on_features(
    feature_list,
    X_train, y_train,
    X_val, y_val,
    cat_cols,
    seed=42
):
    Xtr = X_train[feature_list].copy()
    Xva = X_val[feature_list].copy()

    sub_cat_cols = [c for c in cat_cols if c in feature_list]
    cat_idx = [Xtr.columns.get_loc(c) for c in sub_cat_cols]

    cb = CatBoostClassifier(
        loss_function="MultiClassOneVsAll",
        eval_metric="TotalF1:average=Macro",
        iterations=3000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=10.0,
        random_seed=seed,
        verbose=200,
        auto_class_weights="SqrtBalanced",
        od_type="Iter",
        od_wait=300
    )

    cb.fit(
        Xtr, y_train,
        cat_features=cat_idx,
        eval_set=(Xva, y_val),
        use_best_model=True
    )

    return cb, cat_idx


## Focused Top-K sweep (40/35/30/25/20) + checkpoint saving

Trains one model per K and writes `feature_selection_results_partial.csv` after each K.


In [7]:
# ------------------ sweep settings ------------------
p = len(ranked_features)
K_LIST = [40, 35, 30, 25, 20]
K_LIST = [k for k in K_LIST if 10 < k <= p]
print("Feature counts to test:", K_LIST)

MIN_RECALL_1 = 0.25
MAX_PRED2_RATE = 0.15

rows = []

for K in K_LIST:
    feats = ranked_features[:K]
    print("\n" + "="*70)
    print(f"Training CatBoost with Top-{K} features...")

    cb_model, _ = train_catboost_on_features(
        feats, X_train, y_train, X_val, y_val, cat_cols, seed=RANDOM_SEED
    )

    proba_val = cb_model.predict_proba(X_val[feats])

    # Argmax (reference)
    pred_val_argmax = np.argmax(proba_val, axis=1)
    m_argmax = eval_multiclass(y_val, pred_val_argmax, name=f"CatBoost Top-{K} Argmax - Val")

    # Two-threshold (selection)
    tau1, tau2, _ = tune_two_thresholds(
        proba_val, y_val, min_recall_1=MIN_RECALL_1, max_pred2_rate=MAX_PRED2_RATE
    )
    pred_val_thr = apply_two_thresholds(proba_val, tau1, tau2)
    m_thr = eval_multiclass(y_val, pred_val_thr, name=f"CatBoost Top-{K} Two-Threshold (tau1={tau1:.2f}, tau2={tau2:.2f}) - Val")

    score = m_thr["macro_recall"] + 0.5 * m_thr["precision_high"]

    row = {
        "K": K,
        "score": float(score),
        "tau1": float(tau1),
        "tau2": float(tau2),
        "macro_recall_val": m_thr["macro_recall"],
        "recall1_val": m_thr["recall_1"],
        "recall2_val": m_thr["recall_2"],
        "precision2_val": m_thr["precision_high"],
        "pred2_rate_val": m_thr["pred2_rate"],
        "macro_recall_val_argmax": m_argmax["macro_recall"],
    }
    rows.append(row)

    # Checkpoint save after each K
    partial = pd.DataFrame(rows).sort_values(["score","macro_recall_val"], ascending=False)
    partial_path = MODEL_DIR / "feature_selection_results_partial.csv"
    partial.to_csv(partial_path, index=False)
    print("Checkpoint saved:", partial_path)

results = pd.DataFrame(rows).sort_values(["score","macro_recall_val"], ascending=False).reset_index(drop=True)
display(results)


Feature counts to test: [40, 35, 30, 25, 20]

Training CatBoost with Top-40 features...
0:	learn: 0.4154028	test: 0.4070725	best: 0.4070725 (0)	total: 675ms	remaining: 33m 44s
200:	learn: 0.4467661	test: 0.4262642	best: 0.4268202 (189)	total: 1m 38s	remaining: 22m 44s
400:	learn: 0.4575964	test: 0.4296601	best: 0.4305728 (312)	total: 3m 37s	remaining: 23m 31s
600:	learn: 0.4659212	test: 0.4284784	best: 0.4309037 (507)	total: 5m 40s	remaining: 22m 38s
800:	learn: 0.4727775	test: 0.4254136	best: 0.4309037 (507)	total: 7m 44s	remaining: 21m 14s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.4309036571
bestIteration = 507

Shrink model to first 508 iterations.

=== CatBoost Top-40 Argmax - Val ===
Confusion matrix [0,1,2]:
 [[196137   7126   1528]
 [ 16927   2831   1000]
 [  1855    829    551]]
Metrics: {'macro_recall': 0.4214826709371963, 'recall_high': 0.17032457496136014, 'precision_high': 0.17895420591101008, 'recall_0': 0.9577422835964471, 'recall_1': 0.13638115

,K,score,tau1,tau2,macro_recall_val,recall1_val,recall2_val,precision2_val,pred2_rate_val,macro_recall_val_argmax
0,40,0.577347,0.20,0.125,0.544743,0.434531,0.625348,0.065208,0.135604,0.421483
1,20,0.574715,0.20,0.125,0.543524,0.429184,0.636476,0.062383,0.144267,0.425355
2,30,0.574636,0.25,0.125,0.543190,0.306581,0.632457,0.062892,0.142195,0.427626
3,35,0.574546,0.25,0.125,0.543403,0.299933,0.638640,0.062285,0.144984,0.426118
4,25,0.521828,0.40,0.400,0.491450,0.602370,0.618238,0.060755,0.143887,0.425235


## Choose smallest K within tolerance, retrain chosen model, evaluate Test

- `TOL = 0.01` chooses the smallest K within 0.01 of the best score.


In [8]:
# ------------------------------------------------------------
# Choose smallest K within tolerance of best score
# ------------------------------------------------------------
TOL = 0.01

best_score = results["score"].max()
candidates = results[results["score"] >= (best_score - TOL)].copy()
chosen_row = candidates.sort_values(["K"], ascending=True).iloc[0]
display(chosen_row)

K_chosen = int(chosen_row["K"])
feats_best = ranked_features[:K_chosen]

print("Best score:", float(best_score))
print("Chosen K (smallest within tolerance):", K_chosen)

# Retrain chosen K cleanly (Train -> Val)
cb_best, _ = train_catboost_on_features(
    feats_best, X_train, y_train, X_val, y_val, cat_cols, seed=RANDOM_SEED
)

# Re-tune thresholds on Val for chosen K
proba_val = cb_best.predict_proba(X_val[feats_best])
tau1_best, tau2_best, _ = tune_two_thresholds(
    proba_val, y_val, min_recall_1=MIN_RECALL_1, max_pred2_rate=MAX_PRED2_RATE
)

print("Chosen thresholds:", {"tau1": tau1_best, "tau2": tau2_best})


K                          20.000000
score                       0.574715
tau1                        0.200000
tau2                        0.125000
macro_recall_val            0.543524
recall1_val                 0.429184
recall2_val                 0.636476
precision2_val              0.062383
pred2_rate_val              0.144267
macro_recall_val_argmax     0.425355
Name: 1, dtype: float64

Best score: 0.5773472692366819
Chosen K (smallest within tolerance): 20
0:	learn: 0.3937834	test: 0.3856427	best: 0.3856427 (0)	total: 303ms	remaining: 15m 8s
200:	learn: 0.4469792	test: 0.4341597	best: 0.4345535 (195)	total: 1m 5s	remaining: 15m 10s
400:	learn: 0.4549499	test: 0.4327168	best: 0.4351439 (285)	total: 2m 21s	remaining: 15m 17s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.4351439338
bestIteration = 285

Shrink model to first 286 iterations.
Chosen thresholds: {'tau1': 0.2, 'tau2': 0.125}


In [9]:
# ------------------------------------------------------------
# Test evaluation (REPORT ONLY — do not use to select K)
# ------------------------------------------------------------
proba_test = cb_best.predict_proba(X_test[feats_best])

pred_test_argmax = np.argmax(proba_test, axis=1)
m_test_argmax = eval_multiclass(y_test, pred_test_argmax, name=f"CatBoost Top-{K_chosen} Argmax - Test")

pred_test_thr = apply_two_thresholds(proba_test, tau1_best, tau2_best)
m_test_thr = eval_multiclass(y_test, pred_test_thr, name=f"CatBoost Top-{K_chosen} Two-Threshold (tau1={tau1_best:.2f}, tau2={tau2_best:.2f}) - Test")

print("Predicted class-2 rate:", float((pred_test_thr == 2).mean()))



=== CatBoost Top-20 Argmax - Test ===
Confusion matrix [0,1,2]:
 [[198212   7749   1857]
 [ 16950   3061   1178]
 [  1840    904    667]]
Metrics: {'macro_recall': 0.43126081341895944, 'recall_high': 0.19554382878921137, 'precision_high': 0.1801728795245813, 'recall_0': 0.9537768624469488, 'recall_1': 0.1444617490207183, 'recall_2': 0.19554382878921137, 'pred2_rate': 0.01592819833231505}

=== CatBoost Top-20 Two-Threshold (tau1=0.20, tau2=0.12) - Test ===
Confusion matrix [0,1,2]:
 [[118011  66367  23440]
 [  4466   8940   7783]
 [   222    938   2251]]
Metrics: {'macro_recall': 0.5498994202116642, 'recall_high': 0.6599237760187628, 'precision_high': 0.06724622094760112, 'recall_0': 0.5678574521937465, 'recall_1': 0.42191703242248335, 'recall_2': 0.6599237760187628, 'pred2_rate': 0.14402498945864778}
Predicted class-2 rate: 0.14402498945864778


## Retrain final model on Train+Val and save artifacts


In [10]:
# ============================================================
# Retrain final model on Train + Val with selected features
# ============================================================
X_trval = pd.concat([X_train, X_val], axis=0)
y_trval = pd.concat([y_train, y_val], axis=0)

cb_final, _ = train_catboost_on_features(
    feats_best, X_trval, y_trval, X_val, y_val, cat_cols, seed=RANDOM_SEED
)

OUT_MODEL = MODEL_DIR / f"catboost_selected_top{K_chosen}.cbm"
cb_final.save_model(str(OUT_MODEL))

OUT_THR = MODEL_DIR / f"catboost_selected_top{K_chosen}_thresholds.json"
with open(OUT_THR, "w", encoding="utf-8") as f:
    json.dump({"tau1": float(tau1_best), "tau2": float(tau2_best)}, f, indent=2)

OUT_FEATS = MODEL_DIR / f"catboost_selected_top{K_chosen}_features.txt"
with open(OUT_FEATS, "w", encoding="utf-8") as f:
    for feat in feats_best:
        f.write(feat + "\n")

OUT_RESULTS = MODEL_DIR / "feature_selection_results.csv"
results.to_csv(OUT_RESULTS, index=False)

print("Saved:", OUT_MODEL)
print("Saved:", OUT_THR)
print("Saved:", OUT_FEATS)
print("Saved:", OUT_RESULTS)


0:	learn: 0.3815004	test: 0.3770183	best: 0.3770183 (0)	total: 514ms	remaining: 25m 40s
200:	learn: 0.4449491	test: 0.4445235	best: 0.4448068 (196)	total: 1m 46s	remaining: 24m 48s
400:	learn: 0.4522547	test: 0.4547695	best: 0.4547695 (400)	total: 3m 45s	remaining: 24m 20s
600:	learn: 0.4581008	test: 0.4615188	best: 0.4615188 (600)	total: 5m 44s	remaining: 22m 54s
800:	learn: 0.4631280	test: 0.4677968	best: 0.4677968 (800)	total: 7m 31s	remaining: 20m 38s
1000:	learn: 0.4678891	test: 0.4720743	best: 0.4724556 (937)	total: 9m 21s	remaining: 18m 41s
1200:	learn: 0.4720646	test: 0.4757646	best: 0.4758726 (1193)	total: 11m 21s	remaining: 17m 1s
1400:	learn: 0.4761643	test: 0.4801339	best: 0.4804794 (1397)	total: 13m 28s	remaining: 15m 22s
1600:	learn: 0.4798518	test: 0.4846109	best: 0.4847495 (1582)	total: 15m 48s	remaining: 13m 49s
1800:	learn: 0.4833683	test: 0.4896751	best: 0.4896751 (1800)	total: 18m	remaining: 11m 59s
2000:	learn: 0.4864603	test: 0.4933732	best: 0.4935512 (1992)	total